In [29]:
# from pyspark.sql import SparkSession
import os
import sys
import pyarrow
import json
from pyspark.sql.types import FloatType, DateType, StringType,IntegerType, BooleanType
from pyspark.sql.functions import col,create_map, lit,pandas_udf,from_unixtime,when,max
from shapely.geometry import Point, Polygon
import pandas as pd

In [2]:
jdbc_url = "jdbc:postgresql://localhost:5432/OpenSky"
target_table = 'OpenSky_Aircraft_Bronze'

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r"C:\hadoop\bin"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable



In [1]:
from open_sky_pipeline.Connect import get_spark
spark=get_spark()

Python: c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv\Scripts\python.exe
Version: 3.11.15 (main, Jul 23 2026, 14:42:43) [MSC v.1944 64 bit (AMD64)]
VIRTUAL_ENV: C:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv
SPARK_HOME: None
JAVA_HOME: C:\Program Files\Java\jdk-17.0.2


In [3]:
import os
import sys
import pyspark

print("Python:", sys.executable)
print("PySpark:", pyspark.__version__)
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("PATH Hadoop:", [
    x for x in os.environ["PATH"].split(os.pathsep)
    if "hadoop" in x.lower()
])

print(
    "Hadoop:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Python: c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv\Scripts\python.exe
PySpark: 3.5.3
HADOOP_HOME: C:\hadoop
PATH Hadoop: ['C:\\hadoop\\bin']
Hadoop: 3.3.4


In [2]:
print("Spark:", spark.version)
print(
    "Hadoop:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark: 3.5.3
Hadoop: 3.3.4


In [42]:
df3 = spark.read.parquet("s3a://bronze/aircraft")

In [43]:
max_ingestion=df3.select(max('ingestion_timestamp')).first()[0]

In [44]:
df3=df3.where(col('ingestion_timestamp')==max_ingestion)

In [4]:
df3.inputFiles()

['s3a://bronze/aircraft/part-00000-04929a9f-22b1-4a7a-adf5-7bc6b77cb11f-c000.snappy.parquet',
 's3a://bronze/aircraft/part-00000-3859dafb-1758-4a3e-88ea-e1b83140aa40-c000.snappy.parquet',
 's3a://bronze/aircraft/part-00001-04929a9f-22b1-4a7a-adf5-7bc6b77cb11f-c000.snappy.parquet',
 's3a://bronze/aircraft/part-00001-c5869524-79a1-4236-89ca-2f99801d0779-c000.snappy.parquet']

In [45]:
df3.show()

+------+--------+--------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+--------------------+
|icao24|callsign|origin_country|time_position|last_contact|longitude|latitude|geo_altitude|on_ground|velocity|true_track|vertical_rate|sensors|baro_altitude|squawk|  spi|position_source|category| ingestion_timestamp|
+------+--------+--------------+-------------+------------+---------+--------+------------+---------+--------+----------+-------------+-------+-------------+------+-----+---------------+--------+--------------------+
|39de4e|TVF276D |        France|   1786997728|  1786997728|   -8.023| 34.1814|    12512.04|    false|  210.51|    196.18|            0|   NULL|     11871.96|  7613|false|              0|       0|2026-08-17 22:27:...|
|a5f852|RTY484  | United States|   1786997728|  1786997729|-104.8491| 40.7919|      2628.9|    false|   43.05|    255.47|          2

In [46]:
print(spark.version)

print(spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

3.5.3
3.3.4


In [7]:
print(
    spark.sparkContext._jvm.org.apache.parquet.Version.FULL_VERSION
)

parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)


In [3]:
spark = SparkSession.builder \
    .appName("Test") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .config("spark.driver.memory", "2g") \
    .master("local[*]") \
    .getOrCreate()
    
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
arrow_enabled = spark.conf.get("spark.sql.execution.arrow.pyspark.enabled", "false")

c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [11]:

print(f"PyArrow włączony: {arrow_enabled}")
print(f"[INFO] HADOOP_HOME set to: {os.environ['HADOOP_HOME']}")
print(f"[INFO] PYSPARK_PYTHON set to: {os.environ['PYSPARK_PYTHON']}")
print(f"[INFO] PYSPARK_DRIVER_PYTHON set to: {os.environ['PYSPARK_DRIVER_PYTHON']}")
print(f"Zainstalowana wersja PyArrow: {pyarrow.__version__}")
print(spark.version)

NameError: name 'arrow_enabled' is not defined

In [47]:

int_list=["time_position","last_contact","position_source","category"]
float_list=["longitude","latitude","geo_altitude","velocity","true_track","vertical_rate","baro_altitude"]
bool_list=["on_ground","spi"]

aircraft_dict = {
    0: "No Info",
    1: "Light",
    2: "Small",
    3: "Medium",
    4: "Large",
    5: "High Vortex Large",
    6: "Heavy",
    7: "High Performance",
    8: "Rotorcraft",
    9: "Glider / Sailplane",
    10: "Lighter than Air",
    11: "Parachutist / Skydiver",
    12: "Ultralight / Hang Glider / Paraglider",
    13: "Reserved / Unassigned",
    14: "Unmanned Aerial Vehicle (UAV)",
    15: "Space / Trans-atmospheric Vehicle",
    16: "Surface Vehicle - Emergency Vehicle",
    17: "Surface Vehicle – Service Vehicle",
    18: "Point Obstacle",
    19: "Cluster Obstacle",
    20: "Line Obstacle"
}

position_source_dict = {
    0: "ADS-B",
    1: "ASTERIX",
    2: "MLAT",
    3: "FLARM"
}


aircraft_map = create_map(
    *[item for kv in aircraft_dict.items() for item in (lit(kv[0]), lit(kv[1]))]
)
position_source_map = create_map(
    *[item for kv in position_source_dict.items() for item in (lit(kv[0]), lit(kv[1]))]
)

with open("poland.json", "r") as f:
    Poland_Polygon = json.load(f)
    
poland_polygon = Polygon(
    Poland_Polygon["features"][0]["geometry"]["coordinates"][0]
)


In [48]:
@pandas_udf("boolean")
def check_point_in_polygon(long: pd.Series, lat: pd.Series) -> pd.Series:
    return pd.Series(
        [
            poland_polygon.contains(Point(lon, la))
            for lon, la in zip(long, lat)
        ]
    )


In [49]:
print(
    check_point_in_polygon.func(
        pd.Series([19.0, 21.0]),
        pd.Series([52.0, 50.0])
    )
)

0    True
1    True
dtype: bool


In [7]:

with open("db.json", "r") as f:
    connection_properties = json.load(f)

df = spark.read \
    .jdbc(url=jdbc_url, table=target_table, properties=connection_properties)

In [13]:
df3.select("ingestion_timestamp").distinct().orderBy("ingestion_timestamp").show(5)

+--------------------+
| ingestion_timestamp|
+--------------------+
|2026-08-17 22:26:...|
|2026-08-17 22:27:...|
+--------------------+



In [50]:

df=df3.withColumns({col: df3[col].cast(FloatType()) for col in float_list}) \
  .withColumns({col: df3[col].cast(IntegerType()) for col in int_list}) \
  .withColumns({col: df3[col].cast(BooleanType()) for col in bool_list})

In [51]:
cols=df.columns
cols.remove("ingestion_timestamp")
count_before_enrichment=df.select(cols).distinct().count()

In [ ]:

df=df.withColumn(
        "altitude_diff", 
        col("geo_altitude") - col("baro_altitude")
        )\
    .withColumn(
        "last_contact_h",
        from_unixtime(col("last_contact"))
        )\
    .withColumn(
        "time_position_h",
        from_unixtime(col("time_position"))
        )\
    .withColumn(
        "vertical_category",
        when(col("vertical_rate") > 0, "Climbing")
        .otherwise(when(col("vertical_rate")==0,"Constant Altitude")
        .otherwise("Descending"))
        )\
    .withColumn(
        "aircraft_category",
        aircraft_map.getItem(col("category"))
        )\
    .withColumn(
        "position_source_name",
        position_source_map.getItem(col("position_source"))
        )\
    .withColumn(
        "isPoland",
        check_point_in_polygon(col("longitude"), col("latitude"))
        )
# df=df.withColumn("last_contact_h",from_unixtime(col("last_contact")))
# df=df.withColumn("time_position_h",from_unixtime(col("time_position")))
# df=df.withColumn(
#     "vertical_category",
#     when(col("vertical_rate") > 0, "Climbing")
#     .otherwise(when(col("vertical_rate")==0,"Constant Altitude")
#     .otherwise("Descending"))
#     )
# df = df.withColumn(
#     "aircraft_category",
#     aircraft_map.getItem(col("category"))
# )
# df = df.withColumn(
#     "position_source_name",
#     position_source_map.getItem(col("position_source"))
# )
# df = df.withColumn(
#     "isPoland",
#     check_point_in_polygon(col("longitude"), col("latitude"))
# )


c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv\Lib\site-packages\pyspark\sql\column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


In [53]:
df.select("isPoland").show()

+--------+
|isPoland|
+--------+
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
|   false|
+--------+
only showing top 20 rows



In [18]:

print(sys.version)
print(sys.executable)

3.11.15 (main, Jul 23 2026, 14:42:43) [MSC v.1944 64 bit (AMD64)]
c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv\Scripts\python.exe


In [54]:
cols=df.columns
cols.remove("ingestion_timestamp")
count_after_enrichment=df.select(cols).distinct().count()

In [55]:
print(count_before_enrichment==count_after_enrichment)

True


In [56]:
try:
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save("s3a://silver/aircraft")
except Exception as e:
    print(e)

In [ ]:
df3_hist = spark.read.parquet("s3a://silver/aircraft")

In [ ]:
try:
    df3_hist.write \
        .format("delta") \
        .mode("append") \
        .save("s3a://silver/aircraft_hist")
except Exception as e:
    print(e)

In [55]:
df.select("true_track").orderBy("true_track", ascending=False).show()

+----------+
|true_track|
+----------+
|    359.87|
|    359.87|
|    359.87|
|    359.85|
|    359.84|
|    359.79|
|    359.79|
|    359.77|
|    359.73|
|    359.66|
|    359.62|
|    359.62|
|    359.62|
|    359.61|
|    359.61|
|     359.6|
|    359.58|
|    359.58|
|    359.53|
|    359.52|
+----------+
only showing top 20 rows
